Look at it by hour - Can we infer when someone is actually using hot water vs when it is idle
Subtract idle energy use from actual people use - integrating beneath the curve to get energy 

Is it possible to separate the pump from the thermostat? 
- Fast fourier transform (turn it from a time domain to a frequency domain) 
- (thermostat is a high frequency device) vs heat
    - low frequency events due to a pump (pump)

In [ ]:
import numpy as np 
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math
import random
from datetime import datetime, timedelta
from types import SimpleNamespace

In [ ]:
# grab all daily files and combine
files = glob.glob("POWER_*.csv")
# files = glob.glob("POWER_202607*.csv") + glob.glob("POWER_20260801.csv") # filtering between specific dates

dfs = [pd.read_csv(f) for f in files]
wide = pd.concat(dfs, ignore_index=True)

In [ ]:
# DATE = "2026-07-14"

## 1. Data Cleaning

Reshapes raw power-sensor exports into time series, and measures time gap between consecutive readings.

Each CSV is in wide format, one row per sensor (entity_id, type, unit), and one collumn per timsetamp = Too many columns

As FFT is the goal, has assumption of uniform spacing. 

In [ ]:
# pivot wide -> long
id_cols = ["entity_id", "type", "unit"]

long_parts = []
for f in files:
    df = pd.read_csv(f)
    melted = df.melt(id_vars=id_cols, var_name="timestamp", value_name="power_w")
    long_parts.append(melted)

long = pd.concat(long_parts, ignore_index=True)

n_before = len(long)
long = long.dropna(subset=["power_w"])
n_after = len(long)
print(f"Dropped {n_before - n_after} rows with missing power_w ({(n_before - n_after) / n_before * 100:.3f}%)")

long["power_w"] = pd.to_numeric(long["power_w"])
long["timestamp"] = (
    pd.to_datetime(long["timestamp"], format="mixed", utc=True)
    .dt.tz_convert("Australia/Perth")
)

# clean duplicates
long = long.drop_duplicates(subset=["entity_id", "timestamp"])

# sort by time 
long = long.sort_values(by=['timestamp'])

long["diff_sec"] = (
    long.groupby("entity_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)

Timeframe of data collected

In [ ]:
print(f"First timestamp: {long['timestamp'].min()}")
print(f"Last timestamp:  {long['timestamp'].max()}")
print(f"Total span: {long['timestamp'].max() - long['timestamp'].min()}")

diff_sec appears to have large range of values; min vs mean, reflecting the spliced data of dates that are not contiguous.
Statistical description of spread of data necessary to inspect, for later addressing in data preprocessing for FFT.

In [ ]:
long["diff_sec"].describe()

In [ ]:
gap60 = long[long["diff_sec"].round(0) == 60]
print(f"Count: {len(gap60)}")
print(f"Total missed time: {gap60['diff_sec'].sum() / 60:.1f} minutes")

# observed_time = time actually covered by the sensor
observed_time = long.loc[long["diff_sec"] <= 62, "diff_sec"].sum()
print(f"As % of observed time: {gap60['diff_sec'].sum() / observed_time * 100:.3f}%")

In [ ]:
# imputation - for each row with diff_sec ~60 / 61 seconds, insert a row between the existing row before and itself, to make diff_sec ~30s
long = long.sort_values("timestamp").reset_index(drop=True)
long["diff_sec"] = long["timestamp"].diff().dt.total_seconds()

# find the gaps to fix
gap_mask = long["diff_sec"].between(59, 62)
gap_rows = long[gap_mask]

new_rows = []
for idx in gap_rows.index:
    before = long.loc[idx - 1]
    after = long.loc[idx]

    mid_timestamp = before["timestamp"] + (after["timestamp"] - before["timestamp"]) / 2
    mid_power = (before["power_w"] + after["power_w"]) / 2

    new_row = before.copy()
    new_row["timestamp"] = mid_timestamp
    new_row["power_w"] = mid_power
    new_rows.append(new_row)

# insert the new rows and re-sort
long = pd.concat([long, pd.DataFrame(new_rows)], ignore_index=True)
long = long.sort_values("timestamp").reset_index(drop=True)

# recompute diff_sec now that the gaps are filled
long["diff_sec"] = long["timestamp"].diff().dt.total_seconds()

In [ ]:
print((long["diff_sec"].round(0) == 60).sum())  # should be 0 now
long["diff_sec"].describe()

In [ ]:
big_gap = long[long["diff_sec"] > 60]
big_gap[["timestamp", "diff_sec"]]

# 2 outliers with diff_sec exceeding ~30 seconds

In [ ]:
long = long.sort_values("timestamp").reset_index(drop=True)
long["diff_sec"] = long["timestamp"].diff().dt.total_seconds()

# the two specific gaps you're after
target_gaps = long[long["diff_sec"].round(2).isin([769.96, 72754.99])]

for idx in target_gaps.index:
    before = long.loc[idx - 1, "timestamp"]
    after = long.loc[idx, "timestamp"]
    diff = long.loc[idx, "diff_sec"]
    print(f"Before: {before}  |  After: {after}  |  diff_sec: {diff:.3f}")

# There is a 20 hour gap; split at July 22. Sensor readings went down for a day. For this reason, must treat FFT independently on each segment.

In [ ]:
gap_threshold_sec = 60  # anything bigger than ~2x the expected 30s interval
long['segment_id'] = (long['diff_sec'] > gap_threshold_sec).cumsum()

long.groupby('segment_id').agg(
    start=('timestamp', 'min'),
    end=('timestamp', 'max'),
    n_samples=('timestamp', 'size')
)

In [ ]:
long = long[['power_w', 'timestamp', 'diff_sec', 'segment_id']].copy()
# aliasing 
# nyquist frequency

# long.describe()

In [ ]:
long.head(10)

In [ ]:
# non-intrusive load monitoring (NILM) + signal processing problem 

# disaggregating pump vs thermostat using FFT 
# thermostat (high-frequency)
# pump (low frequency; long, continuous stretch)

# assume 30 second gap + run fft 
# single-day 

## 2. FFT

Nyquist domain - sampling rate must be at least twice the highest frequency component of the signal to avoid aliasing (form of distortion caused by undersampling)

Converts time-domain into frequency-domain.
Which frequencies contribute the most energy to the signal?

In [ ]:
from scipy.fft import rfft, rfftfreq
from scipy.signal import find_peaks


Hypothesis: Water heater's total power draw is 2 devices: pump + thermostat. 
Fourier analysis attempts to separate these by looking at how often each component's signature repeats, rather than by looking at raw power level.

In [ ]:
# fourier analysis: Any signal can be represented by a weighted sum of sine and cosine functions
# fourier transform decomposes signal into its frequency components 

# fil_long = long[long["timestamp"].dt.date == pd.Timestamp(DATE).date()] # Monday 

SAMPLE_SPACING = 30           # seconds
NYQUIST_HZ = 1 / (2 * SAMPLE_SPACING)
fft_results = {}

for seg_id, seg in long.groupby('segment_id'):
    seg = seg.dropna(subset=['power_w'])
    
    # Resample to ensure fixed time step and extract clean array
    seg = seg.sort_values('timestamp')
    seg = seg.set_index('timestamp').resample(f'{SAMPLE_SPACING}s')['power_w'].mean().dropna()
    signal = seg.to_numpy()
    N = len(signal)
    if N < 100:  # skip tiny leftover segments
        continue

    # Remove DC component so low-frequency bins remain clean
    signal = signal - signal.mean()   

    # Perform Real FFT
    spectrum = rfft(signal)
    freqs = rfftfreq(n=N, d=SAMPLE_SPACING)   # 0 ... Nyquist (1/60 Hz)

    amplitude = 2 * np.abs(spectrum) / N      # Watts: amplitude of oscillation at each frequency
    amplitude[0] = 0                          # DC removed
    if N % 2 == 0:
        amplitude[-1] = np.abs(spectrum[-1]) / N  # Nyquist bin has no mirror

    fft_results[seg_id] = {
        'start': seg.index.min(),
        'end': seg.index.max(),
        'hours': (seg.index.max() - seg.index.min()).total_seconds() / 3600,
        'n': N,
        'freqs': freqs,
        'amplitude': amplitude,
    }

    plt.figure(figsize=(10, 4))
    plt.plot(freqs[1:], amplitude[1:])        
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Amplitude (W)')
    start_local = fft_results[seg_id]['start'].tz_convert('Australia/Perth')
    plt.title(f'Amplitude spectrum, Segment {seg_id + 1} '
              f'({start_local:%d %b}, {fft_results[seg_id]["hours"]:.0f} h)')
    
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Highest peak ~0.0065 Hz, amplitude ~30W; period = 1/ 0.0065 = 154s (~2.5 min); heating control loop's modulation period
    # Likely that water heater controls heating power by rapidly flipping heating elements fully on and fully off for fraction-of-a-second bursts
    # resistive wire coil inside water tube is NOT like a dimmable lightbulb 
    # DHE uses closed-loop "4i" control: inlet / outlet temp + flow sensors, motorised valve (https://www.stiebel-eltron.com/en/home/products-solutions/dhw/instantaneous_waterheater/for-professionals.html)
    

In [ ]:
# Baseline: the 0 Hz amplitude equals the mean power over the segment.

baseline_table = pd.DataFrame([
    {
        'segment_id': seg_id,
        'start_perth': r['start'].tz_convert('Australia/Perth'),
        'duration_h': round(r['hours'], 1),
        'mean_power_w (0 Hz)': round(r['amplitude'][0], 2),
    }
    for seg_id, r in fft_results.items()
])
baseline_table

In [ ]:
# Dominant frequency peaks per segment, converted to repeat intervals.

peak_rows = []
for seg_id, r in fft_results.items():
    freqs, amp = r['freqs'][1:], r['amplitude'][1:]   # drop 0 Hz (the mean)
    peaks, _ = find_peaks(amp)
    top_peaks = peaks[np.argsort(amp[peaks])[-10:]][::-1]

    for rank, p in enumerate(top_peaks, start=1):
        f = freqs[p]
        peak_rows.append({
            'segment_id': seg_id,
            'rank': rank,
            'frequency_hz': f,
            'amplitude_w': amp[p],
            'period_sec': 1 / f,
            'period_min': 1 / f / 60,
        })

peak_table = pd.DataFrame(peak_rows)
print(peak_table.round({'frequency_hz': 6, 'amplitude_w': 2, 'period_sec': 1, 'period_min': 2}).to_string(index=False))

# Periods close to 60 s sit near the Nyquist limit; cycles faster than that alias into
# lower frequencies and cannot be resolved with 30 s sampling.

## 3. Statistics

#### Gaussian Mixture Model

In [ ]:
# fil_long = long[long["timestamp"].dt.date == pd.Timestamp(DATE).date()]

power = long['power_w'].dropna()
power_nonzero = power[power > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(power_nonzero, bins=200, kde=True, ax=axes[0])
axes[0].set_title('Power distribution (linear)')
axes[0].set_xlabel('Power [W]')

sns.histplot(power_nonzero, bins=200, kde=True, ax=axes[1], log_scale=True)
axes[1].set_title('Power distribution (log x-axis)')
axes[1].set_xlabel('Power [W]')

plt.tight_layout()
plt.show()

# filter out baseline energy usage from human usage - then fit 
# Was there a power threshold for more than 40 seconds - Yes/No; sliding window 

# Main components: 
# Heating elements; ~1000 - 2500 W+ (broad, far right)
# Microprocessor & control electronics; continuously monitor inlet temp, outlet temp, water flow rate via internal sensors (< 5W in standby mode)
# Motorized flow-control valve (automatically throttles / increases flow if water too cold) - middle peak (~150 - 250W)

# Legionella risk occurs when hot water sists inside copper / PEX supply lines to tap

In [ ]:
from sklearn.mixture import GaussianMixture

# GMM: assumes power readings come from a small number of underlying operating states,
# each approximately Gaussian. Fitted on log10(power) because the states span three orders
# of magnitude and noise scales with power level.
train_data = long.loc[long['power_w'] > 0, 'power_w'].values.reshape(-1, 1)
u_train = np.log10(train_data)

ks = range(1, 7)
models = {k: GaussianMixture(n_components=k, random_state=42, n_init=10).fit(u_train) for k in ks}  # 10 EM restarts, keep best
bic = [models[k].bic(u_train) for k in ks]

for k in ks:
    print(f"k={k}: BIC={bic[k-1]:,.0f}, means={np.sort(10**models[k].means_.ravel()).round(1)} W")

plt.plot(ks, bic, 'o-')
plt.xlabel('Number of clusters k'); plt.ylabel('BIC')
plt.title('Choosing k (log-space fit)'); plt.show()

# hypothesis: 
# 2.9W - 'standby'
# 143.4W - 'modulating heating'  
# 1906.6W - 'heating element active; running at maximum power with the valve throttling flow' 

In [ ]:
from scipy.stats import norm

x = train_data.ravel()
u = np.log10(x)                                        # work in log10(W) from here on
u_train = u.reshape(-1, 1)

gmm3 = GaussianMixture(n_components=3, random_state=42, n_init=10).fit(u_train)

order   = np.argsort(gmm3.means_.ravel())
means   = gmm3.means_.ravel()[order]                   # in log10(W)
stds    = np.sqrt(gmm3.covariances_.ravel()[order])
weights = gmm3.weights_[order]
names   = ['standby', 'intermediate (modulated)', 'heating-element']

us = np.linspace(u.min(), u.max(), 2000)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(u, bins=150, density=True, color='lightgrey', label='power readings (data)')
for i in range(3):
    ax.plot(us, weights[i] * norm.pdf(us, means[i], stds[i]), lw=2,
            label=f'{names[i]}: {10**means[i]:.0f} W ({weights[i]:.0%})')

ax.set_yscale('log'); ax.set_ylim(1e-3, 1e2)
ticks = [1, 3, 10, 30, 100, 300, 1000, 3000]
ax.set_xticks(np.log10(ticks)); ax.set_xticklabels(ticks)
ax.set_xlabel('Power [W]'); ax.set_ylabel('Density (log scale)')
ax.set_title('GMM (k=3) fitted on log10(power)'); ax.legend(); plt.show()

In [ ]:
x = train_data.ravel()
u = np.log10(x)                                        # work in log10(W) from here on
u_train = u.reshape(-1, 1)

gmm4 = GaussianMixture(n_components=4, random_state=42, n_init=10).fit(u_train)

order   = np.argsort(gmm4.means_.ravel())
means   = gmm4.means_.ravel()[order]                   # in log10(W)
stds    = np.sqrt(gmm4.covariances_.ravel()[order])
weights = gmm4.weights_[order]
names   = ['standby', 'intermediate (modulated)', 'recirculation pump', 'heating element']

us = np.linspace(u.min(), u.max(), 2000)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(u, bins=150, density=True, color='lightgrey', label='power readings (data)')
for i in range(4):
    ax.plot(us, weights[i] * norm.pdf(us, means[i], stds[i]), lw=2,
            label=f'{names[i]}: {10**means[i]:.0f} W ({weights[i]:.0%})')

ax.set_yscale('log'); ax.set_ylim(1e-3, 1e2)
ticks = [1, 3, 10, 30, 100, 300, 1000, 3000]
ax.set_xticks(np.log10(ticks)); ax.set_xticklabels(ticks)
ax.set_xlabel('Power [W]'); ax.set_ylabel('Density (log scale)')
ax.set_title('GMM (k=4) fitted on log10(power)'); ax.legend(); plt.show()

In [ ]:
t = long['timestamp'].dt.tz_convert('Australia/Perth')
win = long[(t >= '2026-07-13 00:30') & (t <= '2026-07-13 05:30')]
plt.figure(figsize=(14, 4))
plt.scatter(win['timestamp'].dt.tz_convert('Australia/Perth'), win['power_w'], s=8)
plt.yscale('log'); plt.ylabel('Power [W]'); plt.title('13 July, 00:30–05:30 (Perth)'); plt.show()

### Step 1. Establish the event threshold

In [ ]:
# if power > 1090.4 W for more than 25 seconds = Human use 

spike_threshold_w  = 1090.4
min_duration_sec = 25

In [ ]:
def runs_above(threshold_w):
    """Continuous runs of readings above threshold_w, split at data gaps."""
    df = long.sort_values('timestamp')
    above = df['power_w'] > threshold_w
    new_seg = df['segment_id'] != df['segment_id'].shift()
    run_id = ((above != above.shift()) | new_seg).cumsum()

    runs = df[above].groupby(run_id[above])['timestamp'].agg(start='min', end='max')
    runs['duration_sec'] = (runs['end'] - runs['start']).dt.total_seconds()
    runs['hour'] = runs['start'].dt.tz_convert('Australia/Perth').dt.hour
    runs['qualifying'] = runs['duration_sec'] > min_duration_sec
    return runs


def plot_threshold(threshold_w):
    eb = runs_above(threshold_w)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, q, title in [(axes[0], False, f'(a) Single readings > {threshold_w} W'),
                         (axes[1], True,  f'(b) Qualifying events (≥ 2 consecutive readings > {threshold_w} W)')]:
        eb.loc[eb['qualifying'] == q, 'hour'].value_counts().reindex(range(24), fill_value=0).plot(kind='bar', ax=ax)
        ax.set_title(title); ax.set_xlabel('Hour of day (Perth)'); ax.set_ylabel('Number of runs')
    plt.tight_layout(); plt.show()

    work = eb['hour'].between(7, 16)
    for q, label in [(False, 'Single readings'), (True, 'Qualifying events')]:
        print(f"{label}: {(eb['qualifying'] == q).sum()} runs, "
              f"{work[eb['qualifying'] == q].mean():.0%} start between 07:00 and 16:59")


plot_threshold(1090.4)          # change the value to test others

#### Step 2. Detect and log every human event

In [ ]:
long = long.sort_values('timestamp').reset_index(drop=True)
long['diff_sec'] = long['timestamp'].diff().dt.total_seconds()

# flag each instance as above/below power threshold 
above = long['power_w'] > spike_threshold_w

# continuous run of above/below samples given a unique id 
# (id only changes when above/below state flips, so one run = one id)
new_segment = long['segment_id'] != long['segment_id'].shift()
long['block_id'] = ((above != above.shift()) | new_segment).cumsum()

# for every continuous run that's above threshold, how long does it last?
event_blocks = (
    long[above]
    .groupby('block_id')['timestamp']
    .agg(start='min', end='max')
)
event_blocks['duration_sec'] = (event_blocks['end'] - event_blocks['start']).dt.total_seconds()

# only keep runs that stay above threshold for more than min_duration_sec
qualifying_blocks = event_blocks[event_blocks['duration_sec'] > min_duration_sec].copy()
qualifying_blocks['date'] = qualifying_blocks['start'].dt.tz_convert('Australia/Perth').dt.date

# mark event_start only at the first sample of each qualifying (real, human-length)
long['event_start'] = long['block_id'].map(qualifying_blocks['start'])
long.loc[long['timestamp'] != long['event_start'], 'event_start'] = pd.NaT

print(f"Above-threshold runs: {len(event_blocks)} | "
      f"single-sample (0 s) runs excluded: {(event_blocks['duration_sec'] == 0).sum()} | "
      f"qualifying events: {len(qualifying_blocks)}")

In [ ]:
# Round to nearest whole second and count frequencies
duration_counts = event_blocks['duration_sec'].round().value_counts().sort_index()
print(duration_counts)

In [ ]:
# spread of runs?

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(event_blocks['duration_sec'], bins=50, ax=axes[0])
axes[0].axvline(min_duration_sec, color='red', linestyle='--', label=f'{min_duration_sec}s cutoff')
axes[0].set_title('All above-threshold runs')
axes[0].set_xlabel('Duration (sec)')
axes[0].legend()

sns.histplot(qualifying_blocks['duration_sec'], bins=50, ax=axes[1])
axes[1].set_title('Qualifying (>25s) runs only')
axes[1].set_xlabel('Duration (sec)')

plt.tight_layout()
plt.show()

# my explanation of these 'robotic intervals' 
# Data was gathered using a Tapo smart energy monitoring plug that records power readings at fixed 30-second intervals
# Because the plug polls at fixed time points rather than logging continuously, events are measured in discrete 30-second blocks
# Minor variations in event duration (like 29, 31, 61 are due to small network transmission delays and API polling jitter.

# Measured duration acts as a "bounding window" rather than the exact duration the water tap was open.
# If a person uses the tap for 10–20 seconds inside a single 30-second window, the system records 1 sample above the power threshold, resulting in a recorded duration of 30 seconds
# If a 35-second tap run crosses the boundary between two polling windows, both windows register high power, causing the system to record a 60 second duration.

# Events are heating-element reheats TRIGGERED by hot water draw, i.e. a proxy for tap use,
# not a direct measurement of how long the tap was open.

In [ ]:
# investigating these 'qualifying runs'

qualifying_blocks = qualifying_blocks.copy()

# pull power stats per block from long
power_stats = (
    long[long['power_w'] > spike_threshold_w]
    .groupby('block_id')['power_w']
    .agg(peak_power_w='max', mean_power_w='mean')
)
qualifying_blocks = qualifying_blocks.join(power_stats)

# convert to Perth local time first
qualifying_blocks['start_perth'] = qualifying_blocks['start'].dt.tz_convert('Australia/Perth')
qualifying_blocks['end_perth'] = qualifying_blocks['end'].dt.tz_convert('Australia/Perth')

# now format the Perth-local versions
qualifying_blocks['start_time_of_day'] = qualifying_blocks['start_perth'].dt.strftime('%I:%M:%S %p')
qualifying_blocks['end_time_of_day'] = qualifying_blocks['end_perth'].dt.strftime('%I:%M:%S %p')

qualifying_blocks['duration_bucket'] = pd.cut(
    qualifying_blocks['duration_sec'],
    bins=[0, 45, 75, 105, 135, 165, 195, float('inf')],
    labels=['~30s', '~60s', '~90s', '~120s', '~150s', '~180s', '>180s'],
)

qualifying_blocks_sorted = qualifying_blocks.sort_values('start')[
    ['start', 'start_perth', 'start_time_of_day', 'duration_sec', 'duration_bucket',
     'peak_power_w']
]
qualifying_blocks_sorted

In [ ]:
# investigate the high numbers (60s, 90s, 120s, 150s, 180s)

# Filter directly on numerical duration (>= 90s seconds)
long_events_df = qualifying_blocks_sorted[
    qualifying_blocks_sorted['duration_sec'] >= 88
]

long_events_df

In [ ]:

# -------------------------------------------------------------------------
# 1. BASELINE IDENTIFICATION & DURATION CLEANING
# -------------------------------------------------------------------------
# Extract idle/standby power baseline from GMM (log10 space -> Linear Watts)
standby_w = 10 ** np.sort(gmm3.means_.ravel())[0]

# Bound time intervals to avoid gap distortion
long['dt_sec'] = long['diff_sec'].where(long['diff_sec'] <= 60, SAMPLE_SPACING).fillna(SAMPLE_SPACING)

# -------------------------------------------------------------------------
# 2. POWER & ENERGY DISAGGREGATION
# -------------------------------------------------------------------------
# Calculate net power and energy above standby baseline
long['excess_w'] = (long['power_w'] - standby_w).clip(lower=0)
long['excess_wh'] = long['excess_w'] * long['dt_sec'] / 3600
long['total_wh'] = long['power_w'] * long['dt_sec'] / 3600

# Identify samples belonging to validated human-use event blocks
qualifying_mask = long['block_id'].isin(qualifying_blocks.index)

# Update qualifying_blocks table with event-level Wh
event_energy = long[qualifying_mask].groupby('block_id')['excess_wh'].sum()
qualifying_blocks = qualifying_blocks.drop(columns='human_use_wh', errors='ignore') \
                                     .join(event_energy.rename('human_use_wh'))

# -------------------------------------------------------------------------
# 3. METRIC AGGREGATION
# -------------------------------------------------------------------------
# Time Metrics
total_sec = long['dt_sec'].sum()
total_hours = total_sec / 3600
observed_days = total_sec / 86400

# Energy Totals (Wh -> kWh)
total_kwh = long['total_wh'].sum() / 1000
human_use_kwh = long.loc[qualifying_mask, 'excess_wh'].sum() / 1000
baseline_kwh = total_kwh - human_use_kwh

# Proportions (Energy & Power Shares)
human_pct = (human_use_kwh / total_kwh) * 100
baseline_pct = (baseline_kwh / total_kwh) * 100

# Average Power Metrics (Watts)
overall_mean_power_w = (total_kwh * 1000) / total_hours
human_share_avg_w = (human_use_kwh * 1000) / total_hours
baseline_share_avg_w = (baseline_kwh * 1000) / total_hours

# -------------------------------------------------------------------------
# 4. REPORT OUTPUT
# -------------------------------------------------------------------------
print("=" * 60)
print("             POWER & ENERGY DISAGGREGATION REPORT           ")
print("=" * 60)

print("\n--- 1. TIME ACCOUNTED FOR ---")
print(f"Total Duration:        {observed_days:.2f} days ({total_hours:.1f} hours / {total_sec:,.0f} seconds)")

print("\n--- 2. TOTAL ENERGY CONSUMPTION ---")
print(f"Total Consumption:     {total_kwh:.2f} kWh")
print(f"Daily Average:         {total_kwh / observed_days:.2f} kWh/day")
print(f"Overall Mean Power:    {overall_mean_power_w:.1f} W")

print("\n--- 3. ENERGY & POWER PROPORTIONS ---")
print(f"Human-Use Energy:      {human_use_kwh:.2f} kWh  |  {human_pct:.1f}% of total energy")
print(f"  └─ Equivalent Power: {human_share_avg_w:.1f} W (averaged across full duration)")

print(f"\nBaseline/Standby:      {baseline_kwh:.2f} kWh  |  {baseline_pct:.1f}% of total energy")
print(f"  └─ GMM Standby Mean:   {standby_w:.1f} W (idle electronics baseline)")
print(f"  └─ Equivalent Power: {baseline_share_avg_w:.1f} W (baseline bucket averaged over full duration; includes standby + non-qualifying above-threshold)")
print("=" * 60)

### Step 3. Bin events by time interval

On average, how many events happen in each hours of the day, splift by weekday vs weekends.

In [ ]:
# Uses `events` (from your event_start detection step) and `long` (for the full date range)
events = long.dropna(subset=['event_start']).copy()
events['event_start'] = pd.to_datetime(events['event_start'])
events['event_start_local'] = events['event_start'].dt.tz_convert('Australia/Perth')
events['hour'] = events['event_start_local'].dt.hour
events['date'] = events['event_start_local'].dt.date

# Count events per hour per day
counts_per_day = (
    events.groupby(['date', 'hour'])
    .size()
    .rename('event_count')
    .reset_index()
)

long_local = long.copy()
long_local['timestamp_local'] = long_local['timestamp'].dt.tz_convert('Australia/Perth')
long_local['date'] = long_local['timestamp_local'].dt.date
long_local['hour'] = long_local['timestamp_local'].dt.hour

# every (date, hour) pair where there is at least one raw sensor reading
observed_date_hours = (
    long_local[['date', 'hour']]
    .drop_duplicates()
)

# is_weekend is a property of the date, not of individual events 
observed_date_hours['is_weekend'] = pd.to_datetime(observed_date_hours['date']).dt.dayofweek >= 5

# Left-join event counts onto the full observed grid.
# Any (date, hour) that had sensor data but no event now explicitly gets 0 instead of being absent from the average entirely.
full_counts = observed_date_hours.merge(
    counts_per_day, on=['date', 'hour'], how='left'
)
full_counts['event_count'] = full_counts['event_count'].fillna(0)

# now, mean is a true "expected events per hour across all observed days",
# because quiet hours contribute a 0 instead of being skipped.
avg_per_hour = (
    full_counts.groupby(['hour', 'is_weekend'])['event_count']
    .mean()
    .reset_index()
)

# Still need this in case some (hour, is_weekend) combo never appeared at all in the observed data 
# (e.g. if dataset has zero weekend days with sensor coverage at some particular hour) — same as before.
full_index = pd.MultiIndex.from_product([range(24), [False, True]], names=['hour', 'is_weekend'])
avg_per_hour = (
    avg_per_hour.set_index(['hour', 'is_weekend'])
    .reindex(full_index, fill_value=0)
    .rename(columns={'event_count': 'avg_events_per_hour'})
    .reset_index()
)

# Table for Appendix D
print(wide.round(4).to_string())

### Step 4. Convert counts into ticking probabilities.

In [ ]:
# lambda = average events per hour / 60 minutes
avg_per_hour['lambda_per_min'] = avg_per_hour['avg_events_per_hour'] / 60

# P(at least 1 event in this minute) = 1 - e^(-lambda)
avg_per_hour['p_trigger'] = 1 - np.exp(-avg_per_hour['lambda_per_min'])

# Build two 24-length lookup arrays: one for weekdays, one for weekends
p_weekday = (
    avg_per_hour[avg_per_hour['is_weekend'] == False]
    .set_index('hour')['p_trigger']
    .reindex(range(24), fill_value=0)
    .values
)
p_weekend = (
    avg_per_hour[avg_per_hour['is_weekend'] == True]
    .set_index('hour')['p_trigger']
    .reindex(range(24), fill_value=0)
    .values
)

print(f"\nWeekday probabilities: \n{p_weekday}")
print(f"\nWeekend probabilities: \n{p_weekend}")

# Most weekday early-morning and late-evening hours (1-6am, 6pm-10pm, and almost entire weekend array show 0)

### Step 5. Implement ABM logic

In [ ]:
import random

def water_heater_tick(simulation_time, p_weekday, p_weekend, water_heater):
    """
    Call this once per simulated minute.
    """
    is_weekend = simulation_time.weekday() >= 5
    lookup = p_weekend if is_weekend else p_weekday

    current_hour = simulation_time.hour
    p_trigger = lookup[current_hour]

    if random.random() < p_trigger:
        # A human agent "uses" the water heater this minute
        water_heater.activate_human_use_state()
    else:
        # No human interaction — standard standby rules apply
        water_heater.run_standby_tick()


# Example usage inside your ABM's main loop:
# for simulation_time in simulation_timeline:  # stepping minute by minute
#     water_heater_tick(simulation_time, p_weekday, p_weekend, water_heater)

### Step 6. Sanity Checking.

With the probabilities found, is it possible to reverse-engineer the results to ensure that model is 'accurate' as can be?

In [ ]:
MINUTES_PER_HOUR = 60

def simulate_day(p_lookup, rng):
    """
    Re-uses the exact same per-minute Bernoulli logic as water_heater_tick():
    for each hour, draw 60 uniform(0,1) numbers and count how many fall
    below that hour's p_trigger. Returns a 24-length array of tick-counts.
    """
    # how many tap-use events happened in this hour?
    hourly_counts = np.zeros(24, dtype=int) # array of 24 zeros, representing an hour of the day - hourly_counts[0] = midnight
    for hour in range(24):
        # p is the 'probability that, in any given minute during this hour, a tap-use event starts'
        p = p_lookup[hour] # p_lookup is either p_weekday / p_weekend
        hourly_counts[hour] = (rng.random(MINUTES_PER_HOUR) < p).sum() # compares all 60 random numbers to p to produce array of 60 T/F
    return hourly_counts

# MAPE for report 
# scores how far simulated hourly means are from observed ones
# drops hours with zero observed events
# Good MAPE: <10%
def weighted_mape(observed, simulated):
    mask = observed > 0 # drops hours with zero real events
    weights = observed[mask] / observed[mask].sum() # Greater weight given to peak hours than quiet hours
    pct_errors = np.abs(simulated[mask] - observed[mask]) / observed[mask]
    return (weights * pct_errors).sum() * 100

1. Per-hour comparison: Simulated mean vs emperical rate p_trigger was built from

In [ ]:
# extract real average number of events observed in that hour, from avg_per_hour (Step 3 & 4)
# to create two 24-length arrays of real data: 'on average, how many events happened in hour X' for wkdays and wkends?

observed_weekday = (
    avg_per_hour[avg_per_hour['is_weekend'] == False]
    .set_index('hour')['avg_events_per_hour']
    .reindex(range(24), fill_value=0).values
)
observed_weekend = (
    avg_per_hour[avg_per_hour['is_weekend'] == True]
    .set_index('hour')['avg_events_per_hour']
    .reindex(range(24), fill_value=0).values
)

N_SIMULATED_DAYS = [1000, 5000, 10000, 50000, 100000]

convergence_results = []

for n in N_SIMULATED_DAYS:
    rng = np.random.default_rng(42)  # reset the seed each time so runs are comparable, not just cumulative

    # array of [N_SIMULATED_DAYS x 24]
    sim_weekday = np.array([simulate_day(p_weekday, rng) for _ in range(n)])
    sim_weekend = np.array([simulate_day(p_weekend, rng) for _ in range(n)])

    mape_weekday = weighted_mape(observed_weekday, sim_weekday.mean(axis=0))
    mape_weekend = weighted_mape(observed_weekend, sim_weekend.mean(axis=0))

    convergence_results.append({
        'N': n,
        'weekday_mape': mape_weekday,
        'weekend_mape': mape_weekend,
    })
    print(f"N={n:>7,}: weekday MAPE = {mape_weekday:5.2f}%   weekend MAPE = {mape_weekend:5.2f}%")

convergence_df = pd.DataFrame(convergence_results)

# weekday converges as early N = 5000 and 10,000, comfortably sitting between ~0.8 - 1.5%. N = 50,
# since weekend has extremely sparse data with several near-zero probabilities, likely won't converge unless at extremely high number N

In [ ]:
comparison = pd.DataFrame({
    'hour': range(24),
    'observed_weekday': observed_weekday,
    'simulated_weekday': sim_weekday.mean(axis=0),
    'observed_weekend': observed_weekend,
    'simulated_weekend': sim_weekend.mean(axis=0),
})
# bias_pct = (simulated - observed) / observed * 100
# negative = simulation undershot; produced less events in that hour than really occurred
# serves as diagnostic; where and which direction model is off, hour by hour - no need to include in report 
comparison['weekday_bias_pct'] = (
    (comparison['simulated_weekday'] - comparison['observed_weekday'])
    / comparison['observed_weekday'].replace(0, np.nan) * 100
)
comparison['weekend_bias_pct'] = (
    (comparison['simulated_weekend'] - comparison['observed_weekend'])
    / comparison['observed_weekend'].replace(0, np.nan) * 100
)
print(comparison.round(3).to_string())

In [ ]:
# convergence graph - MAPE. Does MAPE flatten out as N grows? 

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(convergence_df['N'], convergence_df['weekday_mape'], marker='o', label='Weekday MAPE')
ax.plot(convergence_df['N'], convergence_df['weekend_mape'], marker='o', label='Weekend MAPE')
ax.set_xscale('log')
ax.set_xlabel('N simulated days (log scale)')
ax.set_ylabel('Weighted MAPE (%)')
ax.set_title('MAPE convergence as simulation count increases')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# as N grows, both lines converge. 
# with few simulated days, simulated avgs noisy, error high (wkday - 4%, wkdend - 8.5%)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
width = 0.35
hours = np.arange(24)
axes[0].bar(hours - width/2, comparison['observed_weekday'], width, label='Observed')
axes[0].bar(hours + width/2, comparison['simulated_weekday'], width, label='Simulated')
axes[0].set_title('Weekday: events/hour, observed vs simulated')
axes[0].set_xlabel('Hour of day'); axes[0].legend()

axes[1].bar(hours - width/2, comparison['observed_weekend'], width, label='Observed')
axes[1].bar(hours + width/2, comparison['simulated_weekend'], width, label='Simulated')
axes[1].set_title('Weekend: events/hour, observed vs simulated')
axes[1].set_xlabel('Hour of day'); axes[1].legend()
plt.tight_layout(); plt.show()

Full-day totals: Does the simulated distribution match real days, not just the average?

In [ ]:
# --- 2. full-day totals: does the simulated distribution match real days? ---

# coverage per Perth-local day: 30 s sampling -> 2,880 samples on a complete day
samples_per_day = long_local.groupby('date').size()
coverage = (samples_per_day / 2880).clip(upper=1)
good_days = coverage[coverage >= 0.9].index
print(f"Dropped {len(coverage) - len(good_days)} of {len(coverage)} days for <90% coverage")

# daily totals from the full observed grid, so zero-event days are included
actual_daily_totals = full_counts.groupby('date')['event_count'].sum()
actual_daily_totals = actual_daily_totals[actual_daily_totals.index.isin(good_days)]
actual_daily_totals.index = pd.to_datetime(actual_daily_totals.index)
is_weekend_day = actual_daily_totals.index.dayofweek >= 5

actual_weekday_totals = actual_daily_totals[~is_weekend_day].values
actual_weekend_totals = actual_daily_totals[is_weekend_day].values
sim_weekday_totals = sim_weekday.sum(axis=1)
sim_weekend_totals = sim_weekend.sum(axis=1)

# spread check: a Poisson-like simulation has variance ≈ mean.
# Real variance well above mean = real days differ more than the model allows.
for label, actual, sim in [('Weekday', actual_weekday_totals, sim_weekday_totals),
                           ('Weekend', actual_weekend_totals, sim_weekend_totals)]:
    print(f"{label}: actual mean {actual.mean():.2f}, var {actual.var(ddof=1):.2f} "
          f"(n={len(actual)} days) | simulated mean {sim.mean():.2f}, var {sim.var():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, title, actual, sim in [(axes[0], 'Weekday', actual_weekday_totals, sim_weekday_totals),
                               (axes[1], 'Weekend', actual_weekend_totals, sim_weekend_totals)]:
    bins = np.arange(-0.5, max(actual.max(), sim.max()) + 1.5, 1)  # one bar per whole count
    ax.hist(sim, bins=bins, alpha=0.6, density=True, label='Simulated')
    ax.hist(actual, bins=bins, alpha=0.6, density=True, label='Actual')
    ax.set_title(f'{title}: daily event-count distribution')
    ax.set_xlabel('Events per day'); ax.legend()
plt.tight_layout(); plt.show()

# Simulated daily totals are sums of many small per-minute chances, so they are
# approximately Poisson (variance ≈ mean). Near-normal at the weekday mean (~8),
# clearly skewed at the weekend mean (~0.6).

# Weekend rates are based on ~6 events over ~10 weekend days. 
# The simulation reproduces them, 
# but the underlying profile is too sparse to be reliable; 
# weekend results should be treated as indicative only

# observed data is more sparse

### Step 7. Model different scenarios.